# Genre Bias in the Academy Awards: A Statistical Analysis of Oscar Winning Films

**Group Members:** Edison Ayran, Diego Inostroza, Harshini Sangaraju

In [ ]:
# libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import chi2_contingency
from statsmodels.sandbox.stats.runs import runstest_1samp
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Introduction

This project investigates whether the genre of a film influences its probability of winning an Academy Award. The Academy Awards are widely regarded as one of the most prestigious honors in the film industry, and winning an Oscar can significantly impact a film’s cultural recognition, financial success, and the careers of those involved. Because of this influence, it is important to examine whether certain types of films are systematically favored in award outcomes. Using data from the Academy Awards database and the IMDb dataset, this analysis explores whether genres such as Drama or Biography which are often considered “prestige genres,” are more likely to win awards than genres such as Action, Comedy, or Horror. By combining award data with film metadata, this project aims to statistically evaluate whether genre plays a meaningful role in Oscar success.

# Problem Statement

Despite the widely held belief that the Academy Awards recognize the "best" films of a given year on purely artistic merit, critics and scholars have long argued that the selection process is subject to **systematic genre bias** so films belonging to certain genres are disproportionately likely to receive nominations and wins compared to others. Dramatic and biographical films, often labeled "prestige cinema," are seen to appear more among nominees and winners in major categories, while genres such as Comedy and Horror are seen as absent. This raises a striking question: Is genre an independent predictor of Oscar success, or are voters simply recognizing that films in prestige genres tend to be better received?

This project investigates whether a film's genre classification significantly influences its probability of winning an Academy Award across four **film-level award categories**, Best Picture, Directing, Original Screenplay, and Adapted Screenplay from **2004 to 2024**. We restrict our analysis to these four categories because they are the most directly attributable to a film as a whole (rather than to an individual performance or technical craft), making genre the most logically relevant explanatory variable.

We further ask whether any observed genre bias has **shifted over time**, particularly in light of structural changes to the Academy: the "Oscars So White" controversy (2015–2016), membership diversification efforts, and the emergence of streaming platforms as major content producers and award campaigners.

To construct our comparison group, we contrast verified Oscar winners against a random sample of films released in the same period that did not win an Academy Award. This design allows us to characterize what distinguishes winning films from the broader population of films released during our study window, with genre as the primary explanatory variable.

# Relevance and Motivation

The Academy Awards represent the pinnacle of recognition in the film industry, where winning significantly impacts a film’s cultural influence, financial success, and the career trajectories of its creators. Because the Oscars serve as a definitive metric of "excellence," it is crucial to investigate whether this definition is genre-agnostic or subject to systemic institutional bias. There has long been a prevailing belief that "prestige" genres, such as Drama and historical biopics, receive disproportionate representation among winners, while genres like Action, Comedy, and Horror—despite their commercial and critical prevalence—are systematically overlooked in major categories. Investigating this problem is essential because award trends directly shape the industry by influencing which stories production studios choose to fund and which narratives achieve mainstream prominence.

Moreover, the landscape of the film industry has shifted significantly over the last two decades due to the emergence of streaming platforms and major structural reforms within the Academy, such as membership diversification initiatives following the 2015–2016 "Oscars So White" controversy. These changes provide a unique opportunity to evaluate whether the relationship between genre and award success has evolved or if "prestige" cinema maintains a persistent structural advantage in the voting process. By statistically evaluating award outcomes across major film-level categories from 2004 to 2024, this project aims to provide empirical evidence to determine whether the Academy’s voting patterns reflect a genuine shift toward a more genre-diverse recognition of cinematic excellence or if institutional biases remain entrenched.

# Objective

Determine whether movie genre significantly influences the probability of winning an Academy
Award across four major film-level categories (Best Picture, Directing, Original Screenplay,
Adapted Screenplay) from 2004–2024, and test whether this relationship has changed
meaningfully between the pre-reform era (2004–2014) and the post-reform era (2015–2024)
using chi-squared tests of independence, the Wald-Wolfowitz runs test, and logistic
regression with genre indicator variables and genre × year interaction terms.

### Research Question

**Among films nominated for an Academy Award in the categories of Best Picture, Directing, Original Screenplay, and Adapted Screenplay between 2004 and 2024, does a film's genre classification significantly influence its probability of winning, and has this relationship changed over the last two decades?**

More specifically: Among nominated films, those already deemed worthy of consideration by Academy voters are "prestige" genres such as Drama and Biography, systematically over-represented among winners relative to their share of nominations? And if so, has the post-2015 era of Academy membership reform produced a measurable reduction in this genre-based advantage?

By restricting our study population to nominees only, we ensure that every film in our dataset had a realistic chance of winning. This framing separates genre bias in *winning* from genre bias in *nomination* our analysis speaks specifically to what distinguishes winners from non-winning nominees, not what distinguishes Oscar films from the general film landscape.

# Description of Variables

**From `full_data.csv`:**

| Column              | Type    | Description |
|---------------------|---------|-------------|
| `FilmId`            | string  | IMDb tconst (e.g. `tt0081398`) - primary merge key with IMDb datasets |
| `Film`              | string  | Title of the nominated film |
| `Year`              | string  | Ceremony year (e.g. `2004/05`) |
| `CanonicalCategory` | string  | Standardized category name - filtered to our 4 target categories |
| `Category`          | string  | Raw category label as listed in the official Academy database |
| `Winner`            | boolean | True if the film won; NaN treated as False (nominated but did not win) |
| `Name`              | string  | Person associated with the nomination (director or writer) |

**From `title.basics.tsv.gz`:**

| Column           | Type   | Description |
|------------------|--------|-------------|
| `tconst`         | string | IMDb title identifier - merge key |
| `genres`         | string | Comma-separated genre tags, up to 3 per film (e.g. `Drama,Biography,History`) |
| `runtimeMinutes` | float  | Film runtime in minutes - regression control variable |
| `titleType`      | string | Used to pre-filter to movies only before merging |
| `startYear`      | int    | Release year - used to cross-validate ceremony year |

**From `title.ratings.tsv.gz`:**

| Column          | Type   | Description |
|-----------------|--------|-------------|
| `tconst`        | string | IMDb title identifier - merge key |
| `averageRating` | float  | Weighted average IMDb user rating (scale 1–10) |
| `numVotes`      | int    | Total number of user ratings received |

**Engineered during cleaning:**

| Column          | Type   | Description |
|-----------------|--------|-------------|
| `primary_genre` | string | First genre extracted from `genres` - core categorical predictor in all 3 tests |
| `outcome`       | int    | Binary re-encoding of `Winner` - 1 if won, 0 if nominated only |
| `era`           | int    | 0 = pre-reform (2004–2014), 1 = post-reform (2015–2024) |
| `log_votes`     | float  | Log-transformed `numVotes` to reduce right skew before regression |

## Dataset Composition

Our analysis uses three files merged via IMDb tconst identifiers.

**`full_data.csv` (Primary Oscar Source)**
Contains 12,014 nomination records across all Academy Award categories from 1927–2025.
Each record includes a `FilmId` column with an IMDb tconst (e.g. `tt0081398`),
enabling a clean exact join with IMDb
After filtering to our 4 target categories and ceremony years 2004–2024:

| Category                       | Nominations | Winners |
|--------------------------------|-------------|---------|
| BEST PICTURE                   | 171         | 21      |
| DIRECTING                      | 105         | 21      |
| WRITING (Original Screenplay)  | 105         | 21      |
| WRITING (Adapted Screenplay)   | 105         | 21      |
| **Total**                      | **486**     | **84**  |

**`title.basics.tsv.gz` (IMDb)**
Covers ~10 million IMDb titles. Provides genre classifications (up to 3 per film),
runtime in minutes, title type, and release year. Filtered to `titleType == 'movie'`
before merging.

**`title.ratings.tsv.gz` (IMDb)**
Provides weighted average user rating and total vote count for all rated titles.
Joined to basics on `tconst`, then merged with the Oscar records on `FilmId = tconst`.

Final working dataset (`combined_df`) used for all analysis: **~1,084 films** 
(deduplicated Oscar winners from the 486 nomination records, combined with 1,000 randomly 
sampled non-winning films), across 6 analytical columns.

## Load the Data ##

In [ ]:
# Full Oscar dataset 
oscars_raw = pd.read_csv('full_data.csv', sep='\t', low_memory=False)

# IMDb title.basics 
imdb_basics = pd.read_csv(
    'title.basics.tsv.gz',
    sep='\t',
    compression='gzip',
    low_memory=False,
    na_values='\\N'
)

# IMDb title.ratings 
imdb_ratings = pd.read_csv(
    'title.ratings.tsv.gz',
    sep='\t',
    compression='gzip',
    na_values='\\N'
)

print("Oscar records :", oscars_raw.shape)   
print("IMDb basics   :", imdb_basics.shape)
print("IMDb ratings  :", imdb_ratings.shape)

oscars_raw.head()

In [ ]:
imdb_basics.head()

In [ ]:
imdb_ratings.head()

## Filter to 4 Target Categories (edit)

In [ ]:
target_categories = [
    'BEST PICTURE',
    'DIRECTING',
    'WRITING (Original Screenplay)',
    'WRITING (Adapted Screenplay)'
]

oscars_filtered = oscars_raw[oscars_raw['CanonicalCategory'].isin(target_categories)].copy()
print(f"After category filter: {len(oscars_filtered)} records")

oscars_filtered['year_numeric'] = oscars_filtered['Year'].str.split('/').str[0].astype(int)

oscars_filtered = oscars_filtered[
    (oscars_filtered['year_numeric'] >= 2004) & 
    (oscars_filtered['year_numeric'] <= 2024)
].copy()
print(f"After 2004-2024 filter: {len(oscars_filtered)} records")

print("\nNominations by category:")
print(oscars_filtered['CanonicalCategory'].value_counts())
print()

## Merge with IMDb Data (Genres + Ratings) (edit)

In [ ]:
oscars_with_genre = oscars_filtered.merge(
    imdb_basics[['tconst', 'genres', 'runtimeMinutes', 'startYear']],
    left_on='FilmId',
    right_on='tconst',
    how='left'
)
print(f"After genre merge: {len(oscars_with_genre)} records")
print(f"Missing genres: {oscars_with_genre['genres'].isna().sum()}")

oscars_with_genre = oscars_with_genre.merge(
    imdb_ratings[['tconst', 'averageRating', 'numVotes']],
    on='tconst',
    how='left'
)
print(f"After ratings merge: {len(oscars_with_genre)} records")
print(f"Missing ratings: {oscars_with_genre['averageRating'].isna().sum()}")
print()

## Cleaning Dataset (edit)

In [ ]:
df = oscars_with_genre.dropna(subset=['genres']).copy()

print(f"After dropping missing genres: {len(df)} records")

df['outcome'] = df['Winner'].apply(
    lambda x: 1 if x == True or x == 'True' else 0
)

df['primary_genre'] = df['genres'].str.split(',').str[0]

df['era'] = df['year_numeric'].apply(
    lambda x: 0 if x < 2015 else 1
)

df['runtimeMinutes'] = pd.to_numeric(df['runtimeMinutes'], errors='coerce')
df['averageRating'] = pd.to_numeric(df['averageRating'], errors='coerce')
df['numVotes'] = pd.to_numeric(df['numVotes'], errors='coerce')

df['log_votes'] = np.log1p(df['numVotes'].fillna(0))

final_df = df[[
    'year_numeric',
    'Film',
    'CanonicalCategory',
    'Category',
    'outcome',
    'primary_genre',
    'genres',
    'averageRating',
    'numVotes',
    'log_votes',
    'runtimeMinutes',
    'era',
    'FilmId'
]].copy()

final_df.rename(columns={
    'year_numeric': 'Year',
    'CanonicalCategory': 'Award_Category'
}, inplace=True)

final_df = final_df.sort_values(['Year', 'Award_Category']).reset_index(drop=True)

print(f"Final dataset: {len(final_df)} nominations")
print()


## Initial dataset exploration (edit)

In [ ]:
print("Dataset Overview")
print("="*80)
print()

print(f"Total records: {len(final_df)}")
print(f"Number of columns: {len(final_df.columns)}")
print()

print("Columns:")
print(final_df.columns.tolist())
print()

print("Award Outcome Distribution:")
print(final_df['outcome'].value_counts())
print()

print("Award Categories:")
print(final_df['Award_Category'].value_counts())
print()

print(f"Year range: {final_df['Year'].min()} to {final_df['Year'].max()}")
print()

print("Sample of dataset:")
print(final_df.head(10))
print()

print("Genre Distribution (Top 15):")
print(final_df['primary_genre'].value_counts().head(15))

## Merge the Oscars Dataset ##

In [ ]:
winning_films = oscars_raw[oscars_raw['Winner'] == True][oscars_raw['Class'].isin(['Title', 'Writing'])]
winning_films
winning_films_with_genre = winning_films.merge(
    imdb_basics[['tconst', 'genres']],
    left_on='FilmId',
    right_on='tconst',
    how='left'
)
winning_films_with_genre = winning_films_with_genre.merge(
    imdb_ratings[['tconst', 'averageRating']],
    on='tconst',
    how='left'
)
winning_films_with_genre

## Filter and Clean the Merged Dataset ##

In [ ]:
winning_films_in_range = winning_films_with_genre[~winning_films_with_genre['Year'].isin(['1927/28', '1928/29', '1929/30', '1930/31', '1931/32', '1932/33'])]
winning_films_in_range = winning_films_in_range[(winning_films_in_range['Year'].astype(int) >= 2004) & (winning_films_in_range['Year'].astype(int) <= 2024)]

cleaned_winners = winning_films_in_range.drop_duplicates(subset=['Film'], keep='first').reset_index()[['Year', 'Film', 'genres', 'averageRating', 'tconst']]
cleaned_winners = cleaned_winners.dropna()
cleaned_winners

## Filter and Randomly Sample Non-Winners ##

In [ ]:
filtered_imdb = imdb_basics[~imdb_basics['tconst'].isin(cleaned_winners['tconst'].tolist())]
filtered_imdb = filtered_imdb[(filtered_imdb['startYear'] >= 2004) & (filtered_imdb['startYear'] <= 2024)]
filtered_imdb = filtered_imdb[filtered_imdb['titleType'].isin(['movie', 'short'])]

filtered_imdb = filtered_imdb.merge(
    imdb_ratings[['tconst', 'averageRating']],
    on='tconst',
    how='left'
)

filtered_imdb = filtered_imdb[~filtered_imdb['averageRating'].isna()]
filtered_imdb

In [ ]:
random_nonwinners = filtered_imdb.sample(n=1000, replace=False)
random_nonwinners = random_nonwinners.reset_index()
random_nonwinners

In [ ]:
random_nonwinners = random_nonwinners[['startYear', 'primaryTitle', 'genres', 'averageRating', 'tconst']]
random_nonwinners['startYear'] = random_nonwinners['startYear'].astype(int)
random_nonwinners.columns = ['Year', 'Film', 'genres', 'averageRating', 'tconst']
random_nonwinners

In [ ]:
cleaned_winners['winner'] = [True] * cleaned_winners.shape[0]
random_nonwinners['winner'] = [False] * random_nonwinners.shape[0]

combined_df = pd.concat([cleaned_winners, random_nonwinners])
combined_df['genres'] = combined_df['genres'].str.split(',')
combined_df = combined_df.reset_index(drop=True)
combined_df

# **Exploratory Data Analysis (EDA)**

# EDA set-up and Overview

In [ ]:
# ── EDA 1: Dataset Overview ──────────────────────────────────────────────────
print("=== final_df shape ===")
print(final_df.shape)
print()
print("=== Outcome breakdown (0 = non-winner, 1 = winner) ===")
print(final_df['outcome'].value_counts())
print()
print("=== Year range ===")
print(f"Min year: {final_df['Year'].min()}  |  Max year: {final_df['Year'].max()}")
print()
print("=== Missing values per column ===")
print(final_df[['primary_genre', 'averageRating', 'runtimeMinutes']].isna().sum())
print()
final_df.head(10)

The dataset contains 486 nomination records across our four target categories from 2004–2024, of which 84 are winners (outcome = 1) and the remainder are non-winning nominees (outcome = 0). This framing of nominees vs. winners is the most analytically sound comparison since every film in the dataset was explicitly considered by Academy voters, isolating genre as the variable of interest. The ~17% win rate reflects the natural structure of the award (roughly 5 nominees per category per year). No records are missing primary genre values, confirming a clean merge with IMDb metadata.

# Missingness Analysis

In [ ]:
missing = final_df[['primary_genre', 'averageRating', 'runtimeMinutes', 'numVotes', 'log_votes']].isna().sum()
missing_pct = (missing / len(final_df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print("=== Missing Values in Analytical Columns ===")
print(missing_df)
print()

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(missing_df.index, missing_df['Missing %'], color='steelblue', edgecolor='white')
ax.set_title('Missing Value Rate by Column', fontsize=13, fontweight='bold')
ax.set_xlabel('Missing (%)')
ax.axvline(x=5, color='red', linestyle='--', linewidth=1, label='5% threshold')
ax.legend()
plt.tight_layout()
plt.show()

**Missingness Analysis**

`primary_genre` has no missing values, confirming that all 486 nomination records were successfully matched to IMDb genre metadata, a clean merge. `averageRating` and `numVotes` have a small number of missing entries corresponding to films with too few IMDb votes to receive a rating. `runtimeMinutes` has the highest missingness rate, as some films in the IMDb dataset do not have runtime information logged.

**Handling:** Rows with missing `primary_genre` were dropped before any analysis (none in this case). For `averageRating` and `runtimeMinutes`, missing values are handled at the modeling stage, rows missing these covariates are dropped only for the logistic regression where they are used as control variables, preserving the full 486 records for the chi-squared and runs tests which rely only on genre and outcome.

# Outlier Detection

In [ ]:
outlier_cols = ['averageRating', 'runtimeMinutes', 'numVotes']

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, col in zip(axes, outlier_cols):
    data = final_df[col].dropna()
    ax.boxplot(data, vert=True, patch_artist=True,
               boxprops=dict(facecolor='steelblue', color='white'),
               medianprops=dict(color='goldenrod', linewidth=2),
               whiskerprops=dict(color='gray'),
               capprops=dict(color='gray'),
               flierprops=dict(marker='o', color='red', alpha=0.5))
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_ylabel('Value')

plt.suptitle('Outlier Detection — Regression Control Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# IQR-based outlier counts
print("=== IQR-Based Outlier Counts ===")
for col in outlier_cols:
    data = final_df[col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    outliers = data[(data < Q1 - 1.5 * IQR) | (data > Q3 + 1.5 * IQR)]
    print(f"{col}: {len(outliers)} outliers  (Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f})")

print()
print("=== Descriptive Statistics ===")
print(final_df[outlier_cols].describe().round(2))

**Outlier Analysis**

`averageRating` shows a tight distribution (IQR: 7.30–7.90) with only 3 outliers, one notably low-rated film at 5.30 and a couple approaching the ceiling of 8.80. 
Oscar-nominated films overwhelmingly cluster in the upper range of IMDb scores, 
which makes sense given that poorly received films rarely receive nominations. 
All three outliers are retained as genuine observations.

`runtimeMinutes` shows a right skew with some high-end outliers corresponding to notably long films (e.g. epics exceeding 3 hours). These are legitimate data points, long runtimes are not uncommon among prestige nominees and are retained as-is.

`numVotes` is the most heavily skewed variable, with a long right tail driven by a small number of films with extremely high vote counts. This is expected: some nominated films are major commercial releases with massive audiences while others are smaller independent films. For this reason `numVotes` is log-transformed into `log_votes` before entering the regression model, which substantially compresses the range and reduces the influence of high-vote outliers on coefficient estimates.

No records are removed on the basis of outliers. All values are plausible within the context of Oscar-nominated films and removing them would risk discarding meaningful signal about the relationship between genre, quality, and award outcomes.

# Genre Distribution (All Nominees)

In [ ]:
genre_counts = final_df['primary_genre'].value_counts().head(12)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(genre_counts.index, genre_counts.values, color='steelblue', edgecolor='white')
ax.set_title('Genre Distribution Among All Nominees (2004–2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Primary Genre')
ax.set_ylabel('Number of Nominations')
ax.tick_params(axis='x', rotation=30)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

Drama is the most frequently nominated primary genre by a substantial margin, followed at a distance by Biography, Comedy, and Animation. This distribution reflects the composition of the nominee pool itself before any outcome information is considered. The heavy concentration of Drama nominations raises the question of whether Drama's dominance among winners is proportional to its dominance among nominees, or whether it wins at a rate above what its share of nominations would predict. The charts below address this directly.

# Genre Distribution (Winners Only)

In [ ]:
winners_df = final_df[final_df['outcome'] == 1]
winner_genre_counts = winners_df['primary_genre'].value_counts().head(12)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(winner_genre_counts.index, winner_genre_counts.values, color='goldenrod', edgecolor='white')
ax.set_title('Genre Distribution Among Oscar Winners (2004–2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Primary Genre')
ax.set_ylabel('Number of Winning Films')
ax.tick_params(axis='x', rotation=30)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

Among the 84 winning nominations, Drama's dominance becomes even more concentrated. The winner distribution is noticeably less diverse than the full nominee pool, a handful of genres account for nearly all wins. Genres such as Comedy, Horror, and Action, which appear among nominees, are largely or entirely absent from the winner column. This visual contrast between the all-nominees chart and the winners-only chart provides the first direct evidence of a genre-based disparity in Oscar outcomes.

# Win Rate by Genre

In [ ]:
genre_totals = final_df.groupby('primary_genre').size()
genre_wins   = final_df[final_df['outcome'] == 1].groupby('primary_genre').size()

win_rate = (genre_wins / genre_totals).dropna().sort_values(ascending=False)
win_rate = win_rate[genre_totals[win_rate.index] >= 5]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['goldenrod' if g == 'Drama' else 'steelblue' for g in win_rate.index]
bars = ax.bar(win_rate.index, win_rate.values * 100, color=colors, edgecolor='white')
ax.set_title('Win Rate by Primary Genre (min. 5 nominations, 2004–2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Primary Genre')
ax.set_ylabel('Win Rate (%)')
ax.tick_params(axis='x', rotation=35)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
plt.tight_layout()
plt.show()

print(win_rate.apply(lambda x: f"{x*100:.1f}%"))

Normalizing wins by total nominations per genre reveals that Drama's dominance is not simply a function of volume, it converts nominations to wins at a higher rate than most other genres. This is the key distinction: Drama is over-represented among winners *relative to its share of nominations*, which is the operational definition of genre bias in this context. Genres like Comedy and Animation appear in the nominee pool in meaningful numbers but win at substantially lower rates. This normalized view is what motivates our chi-squared test, we want to formally assess whether these differences in conversion rate are statistically significant or attributable to chance.

# Stacked Comparison: Winners vs Non-Winners by Genre 

In [ ]:
top_genres = genre_totals.nlargest(10).index
plot_data  = final_df[final_df['primary_genre'].isin(top_genres)].copy()
ct = plot_data.groupby(['primary_genre', 'outcome']).size().unstack(fill_value=0)
ct.columns = ['Non-Winner', 'Winner']
ct = ct.loc[top_genres]

ct.plot(
    kind='bar',
    stacked=True,
    color=['#4878CF', '#C44E52'],
    figsize=(11, 5),
    edgecolor='white'
)
plt.title('Winner vs. Non-Winner Counts by Genre (Top 10, 2004–2024)', fontsize=14, fontweight='bold')
plt.xlabel('Primary Genre')
plt.ylabel('Number of Nominations')
plt.xticks(rotation=30)
plt.legend(title='Outcome')
plt.tight_layout()
plt.show()

The stacked bar chart makes the structural disparity concrete. For Drama, the winner segment (red) constitutes a visibly larger proportion of the total bar compared to nearly every other genre, where the non-winner segment (blue) dominates almost entirely. This is not a sample size issue, genres like Biography and Comedy have enough nominations to be represented, yet their winner segments are thin. The chart confirms that the gap between nomination frequency and win frequency varies substantially by genre, and that this variation is not randomly distributed across categories.

# Temporal Trend: Drama Share of Winners by Period 


In [ ]:
final_df['Period'] = pd.cut(
    final_df['Year'],
    bins=[2003, 2009, 2014, 2019, 2024],
    labels=['2004–2009', '2010–2014', '2015–2019', '2020–2024']
)

period_wins = final_df[final_df['outcome'] == 1].groupby(
    ['Period', 'primary_genre'], observed=True
).size().unstack(fill_value=0)

drama_share = (period_wins.get('Drama', 0) / period_wins.sum(axis=1) * 100).dropna()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(drama_share.index.astype(str), drama_share.values, marker='o', color='goldenrod', linewidth=2.5)
ax.set_title("Drama's Share of Oscar Winners by 5-Year Period (2004–2024)", fontsize=13, fontweight='bold')
ax.set_xlabel('Period')
ax.set_ylabel('Drama Win Share (%)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.set_ylim(0, 100)
for x, y in zip(drama_share.index.astype(str), drama_share.values):
    ax.annotate(f"{y:.0f}%", (x, y), textcoords="offset points", xytext=(0, 8), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

Tracking Drama's share of winners across four five-year periods shows that its dominance is persistent but not perfectly stable. While fluctuation exists across periods, there is no clear sustained downward trend following the Academy's post-2015 membership reforms. If anything, the pattern suggests that structural changes to the voting body have not yet produced a measurable reduction in Drama's advantage within our study window. This motivates the inclusion of the `era` variable (0 = pre-2015, 1 = post-2015) in our logistic regression and the Wald-Wolfowitz runs test to formally assess whether any temporal shift is statistically detectable.

# IMDb Rating Distribution: Winners vs Non-Winners 


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for label, color, subset in [
    ('Non-Winner', '#4878CF', final_df[final_df['outcome'] == 0]),
    ('Winner',     '#C44E52', final_df[final_df['outcome'] == 1])
]:
    subset['averageRating'].dropna().plot.kde(ax=ax, label=label, color=color, linewidth=2)

ax.set_title('IMDb Rating Distribution: Winners vs. Non-Winners (2004–2024)', fontsize=13, fontweight='bold')
ax.set_xlabel('IMDb Average Rating')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

print("=== IMDb Rating Summary by Outcome ===")
print(final_df.groupby('outcome')['averageRating'].describe().round(2))

The KDE plot shows that Oscar-winning nominations are associated with modestly higher IMDb user ratings compared to non-winning nominees, with the winner distribution shifted slightly to the right. This overlap is analytically important: it indicates that perceived quality, as proxied by audience ratings, is partially confounded with winning status. Because Drama films may also tend to be rated highly independent of their genre, including `averageRating` as a control variable in our logistic regression is necessary to isolate the effect of genre on winning probability after accounting for overall film quality.

## EDA Summary

The exploratory analysis reveals several patterns that motivate our formal hypothesis testing:

1. **Drama dominates among winners.** Drama is the most prevalent primary genre among Oscar-winning nominations across 2004–2024, and the win-rate plot confirms it converts nominations to wins at a higher rate than most other genres in the dataset.

2. **Comedy, Horror, and Action are structurally underrepresented.** These genres appear among nominees but are largely absent from the winner column, a disparity the stacked bar chart makes visually clear.

3. **The Drama advantage has not clearly declined over time.** Drama's share of winners fluctuates across five-year periods but shows no sustained downward trend, suggesting Academy membership reforms have not fully eliminated the prestige-genre advantage through 2024.

4. **Winners receive moderately higher IMDb ratings.** Oscar-winning nominations cluster slightly higher on IMDb scores, confirming that film quality partially confounds the genre-outcome relationship. This motivates the inclusion of `averageRating` as a control variable in our logistic regression, where `outcome` (1 = winner, 0 = non-winner) is predicted from genre indicator variables and film-level covariates.

## One Hot Encode the Genres ##

In [ ]:
one_hot = pd.get_dummies(combined_df.explode('genres'), columns=['genres'])
one_hot = one_hot.groupby('tconst').max().reset_index(drop=True)
one_hot

## Statistical Analyses

This section conducts three statistical tests to investigate genre bias in Academy Award outcomes using our 486 nominations dataset (2004-2024).

### Test 1: Chi-Squared Test of Independence

**Research Question**: Is there a statistically significant association between a film's primary genre and whether it wins an Academy Award?

**Null Hypothesis** (H0): Genre and winning are independent

**Alternative Hypothesis** (H1): Genre and winning are associated

In [ ]:
# Count nominations by genre
genre_counts = final_df['primary_genre'].value_counts()
print("Nominations by genre:")
print(genre_counts)
print()

# Select genres with at least 10 nominations (sufficient for chi-squared)
min_nominations = 10
significant_genres = genre_counts[genre_counts >= min_nominations].index.tolist()

print(f"Genres with >= {min_nominations} nominations (included in test):")
print(significant_genres)
print()

# Filter dataset to significant genres
chi_df = final_df[final_df['primary_genre'].isin(significant_genres)].copy()

print(f"Dataset for chi-squared test: {len(chi_df)} nominations")
print(f"Winners: {chi_df['outcome'].sum()}")
print(f"Non-winners: {len(chi_df) - chi_df['outcome'].sum()}")
print()

In [ ]:
# Create contingency table: rows = genre, columns = outcome (0=lost, 1=won)
contingency_table = pd.crosstab(
    chi_df['primary_genre'], 
    chi_df['outcome'],
    margins=True
)
contingency_table.columns = ['Lost', 'Won', 'Total']
contingency_table.index.name = 'Genre'
print(contingency_table)
print()

# Calculate win rates by genre
win_rates = chi_df.groupby('primary_genre')['outcome'].agg(['sum', 'count', 'mean'])
win_rates.columns = ['Wins', 'Nominations', 'Win_Rate']
win_rates['Win_Rate_Pct'] = (win_rates['Win_Rate'] * 100).round(1)
win_rates = win_rates.sort_values('Win_Rate', ascending=False)

print("\nWin Rates by Genre:")
print(win_rates)
print()

**Chi-squared test assumptions**:
1. Independence: Each nomination is independent
2. Expected frequencies: All cells should have expected count >= 5

Lets check if these hold:

In [ ]:
# Calculate expected frequencies
# Remove 'All' row/column for chi2 calculation
contingency_no_margins = pd.crosstab(chi_df['primary_genre'], chi_df['outcome'])
chi2_stat, p_value, dof, expected_freq = chi2_contingency(contingency_no_margins)

# Create expected frequency table
expected_df = pd.DataFrame(
    expected_freq,
    index=contingency_no_margins.index,
    columns=['Lost (expected)', 'Won (expected)']
)

print("Expected Frequencies:")
print(expected_df.round(2))
print()

# Check if all expected frequencies >= 5
min_expected = expected_freq.min()
all_valid = min_expected >= 5

print(f"Minimum expected frequency: {min_expected:.2f}")
if all_valid:
    print("All expected frequencies >= 5: Chi-squared test is VALID")
else:
    print("Some expected frequencies < 5: Chi-squared may be unreliable")
    print("Consider combining small categories or using Fisher's exact test")
print()

In [ ]:
#Conducting the chi-squared test
print(f"Chi-squared statistic: {chi2_stat:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"P-value: {p_value:.6f}")
print()

# Interpret results
alpha = 0.05
if p_value < alpha:
    print(f"P-value ({p_value:.6f}) < alpha ({alpha})")
    print("  Result: REJECT null hypothesis")
    print("  Conclusion: There IS a statistically significant association between genre and winning an Academy Award.")
else:
    print(f"P-value ({p_value:.6f}) >= alpha ({alpha})")
    print("  Result: FAIL TO REJECT null hypothesis")
    print("  Conclusion: There is NO statistically significant association between genre and winning an Academy Award.")
print()
# Calculate effect size (Cramér's V)
n = len(chi_df)
cramers_v = np.sqrt(chi2_stat / (n * (min(contingency_no_margins.shape) - 1)))
print(f"Effect size (Cramér's V): {cramers_v:.4f}")

if cramers_v < 0.1:
    effect_interpretation = "negligible"
elif cramers_v < 0.3:
    effect_interpretation = "small"
elif cramers_v < 0.5:
    effect_interpretation = "medium"
else:
    effect_interpretation = "large"
    
print(f"Interpretation: {effect_interpretation} effect size")
print()

In [ ]:
# Plot observed vs expected win rates
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left plot: Observed win rates
ax1 = axes[0]
observed_rates = win_rates.sort_values('Win_Rate_Pct', ascending=True)
bars1 = ax1.barh(observed_rates.index, observed_rates['Win_Rate_Pct'], color='steelblue')

# Add value labels
for i, (idx, row) in enumerate(observed_rates.iterrows()):
    ax1.text(row['Win_Rate_Pct'] + 0.5, i, 
             f"{row['Win_Rate_Pct']:.1f}%",
             va='center', fontsize=10)

overall_win_rate = (chi_df['outcome'].mean() * 100)
ax1.axvline(overall_win_rate, color='red', linestyle='--', linewidth=2,
            label=f'Overall: {overall_win_rate:.1f}%')

ax1.set_xlabel('Win Rate (%)', fontsize=12)
ax1.set_ylabel('Genre', fontsize=12)
ax1.set_title('Observed Win Rates by Genre', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(axis='x', alpha=0.3)

# Right plot: Observed vs Expected counts
ax2 = axes[1]
x = np.arange(len(contingency_no_margins))
width = 0.35

observed_wins = contingency_no_margins[1].values
expected_wins = expected_df['Won (expected)'].values

bars1 = ax2.bar(x - width/2, observed_wins, width, label='Observed', color='steelblue')
bars2 = ax2.bar(x + width/2, expected_wins, width, label='Expected', color='coral')

ax2.set_xlabel('Genre', fontsize=12)
ax2.set_ylabel('Number of Winners', fontsize=12)
ax2.set_title('Observed vs Expected Winner Counts', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(contingency_no_margins.index, rotation=45, ha='right')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print()

### Interpretation: Chi-Squared Test of Independence

The Chi-squared test of independence was utilized to determine if a statistically significant relationship exists between a film's primary genre and its success in winning an Academy Award. The test yielded a **p-value of 0.097**, which is greater than the standard alpha level of 0.05. Consequently, we **fail to reject the null hypothesis**, indicating that there is no statistically significant association between a film's genre and its likelihood of winning an Oscar within our sample.

While the exploratory data analysis revealed that genres like Drama and Biography dominate the raw counts of total nominations and wins, the Chi-squared results demonstrate that their actual win rates (16.7% and 17.0%, respectively) closely align with expected frequencies. This suggests that the high volume of "prestige" winners is largely reflective of the high volume of these films being produced or nominated, rather than a statistically significant proportional advantage in the final voting stage. 

> **Note:** One expected frequency (Adventure) fell marginally below 5 at 4.99, but this minor deviation is unlikely to invalidate the overall non-significant finding as it is essentially negligible.

### Test 2: Wald-Wolfowitz Runs Test

**Research Question**: Are Oscar winners randomly distributed across the pre-reform (2004-2014) and post-reform (2015-2024) eras, or do they cluster in specific periods?

**Null Hypothesis** (H0): Winners are randomly distributed across 
pre/post-reform eras

**Alternative Hypothesis** (H1): Winners cluster in specific eras

In [ ]:
# Create chronological sequence of all nominations
runs_df = final_df.sort_values(['Year', 'Award_Category']).copy()

# Create binary sequence: 1 = pre-reform (2004-2014), 0 = post-reform (2015-2024)
# Focus only on WINNERS to test if they cluster in one era
winners_only = runs_df[runs_df['outcome'] == 1].copy()

print(f"Total winners in sequence: {len(winners_only)}")
print(f"Pre-reform winners (2004-2014): {(winners_only['era'] == 0).sum()}")
print(f"Post-reform winners (2015-2024): {(winners_only['era'] == 1).sum()}")
print()

# Create binary sequence (0 or 1 for era)
sequence = winners_only['era'].values

print("Sample of sequence (first 20 winners):")
print("0 = pre-reform (2004-2014), 1 = post-reform (2015-2024)")
print(sequence[:20])
print()

**Runs Test Assumptions:**
1. Data is sequential/ordered: Winners sorted chronologically
2. Binary categories: Era 0 vs Era 1
3. Each observation is independent

In [ ]:
# Count runs (consecutive sequences of same value)
runs = 1
for i in range(1, len(sequence)):
    if sequence[i] != sequence[i-1]:
        runs += 1

print(f"Number of runs observed: {runs}")
print()

In [ ]:
# Perform runs test
# runstest_1samp returns (z_stat, p_value)
z_stat, p_value = runstest_1samp(sequence, correction=True)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.6f}")
print()

# Interpret results
alpha = 0.05
if p_value < alpha:
    print(f"P-value ({p_value:.6f}) < alpha ({alpha})")
    print("Result: REJECT null hypothesis")
    if z_stat < 0:
        print("Conclusion: Winners show significant CLUSTERING (fewer runs than expected). This suggests winners are concentrated in specific time periods.")
    else:
        print("Conclusion: Winners show significant OSCILLATION (more runs than expected). This suggests winners alternate between eras more than random.")
else:
    print(f"P-value ({p_value:.6f}) >= alpha ({alpha})")
    print("Result: FAIL TO REJECT null hypothesis")
    print("Conclusion: Winners are randomly distributed across eras. No evidence of temporal clustering.")
print()

In [ ]:
# Create timeline plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Top plot: All winners over time
ax1 = axes[0]
years = winners_only['Year'].values
colors = ['coral' if era == 0 else 'skyblue' for era in sequence]

ax1.scatter(years, np.ones(len(years)), c=colors, s=100, alpha=0.6, edgecolors='black')
ax1.set_ylim(0.5, 1.5)
ax1.set_ylabel('')
ax1.set_yticks([])
ax1.set_xlabel('Year', fontsize=12)
ax1.set_title('Timeline of Oscar Winners by Era', fontsize=14, fontweight='bold')
ax1.axvline(2014.5, color='black', linestyle='--', linewidth=2, label='Reform Period')
ax1.legend(loc='upper right')
ax1.grid(axis='x', alpha=0.3)

# Add era labels
ax1.text(2009, 1.3, 'Pre-Reform\n(2004-2014)', ha='center', fontsize=11, 
         bbox=dict(boxstyle='round', facecolor='coral', alpha=0.3))
ax1.text(2019, 1.3, 'Post-Reform\n(2015-2024)', ha='center', fontsize=11,
         bbox=dict(boxstyle='round', facecolor='skyblue', alpha=0.3))

# Bottom plot: Winners per year by era
ax2 = axes[1]
winners_by_year_era = winners_only.groupby(['Year', 'era']).size().unstack(fill_value=0)

if 0 in winners_by_year_era.columns and 1 in winners_by_year_era.columns:
    ax2.bar(winners_by_year_era.index, winners_by_year_era[0], 
            label='Pre-Reform', color='coral', alpha=0.7)
    ax2.bar(winners_by_year_era.index, winners_by_year_era[1], 
            bottom=winners_by_year_era[0],
            label='Post-Reform', color='skyblue', alpha=0.7)
else:
    # Handle case where one era might not exist in some years
    ax2.bar(winners_by_year_era.index, winners_by_year_era.sum(axis=1), 
            color='gray', alpha=0.7)

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Number of Winners', fontsize=12)
ax2.set_title('Winner Distribution Across Years', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)
ax2.axvline(2014.5, color='black', linestyle='--', linewidth=2)

plt.tight_layout()
plt.show()

print()

## Test 3: Logistic Regression

**Research Question:** Which factors predict whether a film wins an Academy Award? Does genre remain significant after controlling for film quality, popularity, and temporal factors?

In [ ]:
# Use genres with sufficient sample size
logit_df = final_df[final_df['primary_genre'].isin(significant_genres)].copy()

print(f"Dataset for logistic regression: {len(logit_df)} nominations")
print(f"Winners: {logit_df['outcome'].sum()}")
print(f"Non-winners: {len(logit_df) - logit_df['outcome'].sum()}")
print()

# Check for missing values in predictor variables
print("Missing values in predictors:")
predictors = ['primary_genre', 'averageRating', 'log_votes', 'runtimeMinutes', 'era']
missing_counts = logit_df[predictors].isnull().sum()
print(missing_counts)
print()

# Drop rows with missing values in predictors
logit_df_clean = logit_df.dropna(subset=predictors).copy()
print(f"After removing missing values: {len(logit_df_clean)} observations")
print()

print("Converting numeric columns to proper types...")
logit_df_clean['averageRating'] = pd.to_numeric(logit_df_clean['averageRating'], errors='coerce')
logit_df_clean['log_votes'] = pd.to_numeric(logit_df_clean['log_votes'], errors='coerce')
logit_df_clean['runtimeMinutes'] = pd.to_numeric(logit_df_clean['runtimeMinutes'], errors='coerce')
logit_df_clean['era'] = pd.to_numeric(logit_df_clean['era'], errors='coerce')
logit_df_clean['outcome'] = pd.to_numeric(logit_df_clean['outcome'], errors='coerce')

# Drop any rows that became NaN after conversion
logit_df_clean = logit_df_clean.dropna(subset=predictors + ['outcome']).copy()
print(f"After ensuring numeric types: {len(logit_df_clean)} observations")
print()

**Logistic Regression Assumptions:**
1. Binary outcome: outcome is 0 (lost) or 1 (won)
2. Independence: Each nomination is independent
3. Linearity: Continuous predictors have linear relationship with log-odds
4. No perfect multicollinearity: Check correlation between predictors
5. Large sample size: We have 482 observations

In [ ]:
# Check multicollinearity
print("Correlation matrix of continuous predictors:")
cont_predictors = ['averageRating', 'log_votes', 'runtimeMinutes', 'era']
corr_matrix = logit_df_clean[cont_predictors].corr()
print(corr_matrix.round(3))
print()

# Visualize correlations
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Continuous Predictors', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Check for high correlations (>0.7)
high_corr = (corr_matrix.abs() > 0.7) & (corr_matrix.abs() < 1.0)
if high_corr.any().any():
    print("Warning: High correlations detected (>0.7)")
    print("This may indicate multicollinearity issues")
else:
    print("No severe multicollinearity detected (all correlations < 0.7)")
print()

In [ ]:
# Create dummy variables for genre (most common genre as reference)
reference_genre = logit_df_clean['primary_genre'].value_counts().index[0]
print(f"Reference category (baseline): {reference_genre}")
print()

# Create dummy variables, dropping the reference category
genre_dummies = pd.get_dummies(logit_df_clean['primary_genre'], prefix='genre', drop_first=False)

# Drop the reference genre column
if f'genre_{reference_genre}' in genre_dummies.columns:
    genre_dummies = genre_dummies.drop(f'genre_{reference_genre}', axis=1)

print(f"Genre dummy variables created: {list(genre_dummies.columns)}")
print()

#Convert boolean dummies to integers (0/1)
genre_dummies = genre_dummies.astype(int)

# Combine all predictors (while ensuring all are numeric values)
X_predictors = pd.DataFrame({
    'averageRating': logit_df_clean['averageRating'].values,
    'log_votes': logit_df_clean['log_votes'].values,
    'runtimeMinutes': logit_df_clean['runtimeMinutes'].values,
    'era': logit_df_clean['era'].values
})

# Concatenate genre dummies
X = pd.concat([genre_dummies.reset_index(drop=True), X_predictors.reset_index(drop=True)], axis=1)

print("Checking all predictor types:")
print(X.dtypes)
print()

# Verify no object or bool types
problematic_types = (X.dtypes == 'object') | (X.dtypes == 'bool')
if problematic_types.any():
    print("ERROR: Some columns have object or bool type!")
    print("Columns with problematic types:", X.columns[problematic_types].tolist())
    print("\nAttempting to fix...")
    
    # Convert all to numeric
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors='coerce')
    
    print("After conversion:")
    print(X.dtypes)
    print()
else:
    print("All columns are numeric!")
    print()

# Get outcome variable
y = logit_df_clean['outcome'].values

# Add constant term
X_with_const = sm.add_constant(X, has_constant='add')

print("Final predictor matrix shape:", X_with_const.shape)
print("Final predictor types:")
print(X_with_const.dtypes.value_counts())
print()
print("Outcome vector shape:", y.shape)
print("Outcome type:", y.dtype)
print()

# Fit the model
print("Fitting logistic regression model...")
model = sm.Logit(y, X_with_const)
result = model.fit(disp=0)
print("Model fitted successfully!")
print()


In [ ]:
#Results and Interpretation
print(result.summary())
print()

# Extract coefficients, p-values, and confidence intervals
coef_df = pd.DataFrame({
    'Coefficient': result.params,
    'Std Error': result.bse,
    'z-value': result.tvalues,
    'P-value': result.pvalues,
    'Odds Ratio': np.exp(result.params),
    '95% CI Lower': np.exp(result.conf_int()[0]),
    '95% CI Upper': np.exp(result.conf_int()[1])
})

# Sort by odds ratio
coef_df_sorted = coef_df.drop('const', errors='ignore').sort_values('Odds Ratio', ascending=False)

print("\nCoefficient Interpretation (Odds Ratios):")
print("="*80)
print(coef_df_sorted.round(4))
print()

# Interpret significant predictors
print("\nInterpretation of Significant Predictors (α = 0.05):")
print("-"*80)

significant_vars = coef_df[coef_df['P-value'] < 0.05].drop('const', errors='ignore')

if len(significant_vars) == 0:
    print("No predictors are statistically significant at α = 0.05")
    print("This suggests that after controlling for all factors,")
    print("none individually predict winning probability.")
    print()
    print("Possible interpretations:")
    print("- The predictors may collectively explain winning (see Pseudo R²)")
    print("- Sample size may be insufficient for individual significance")
    print("- True effects may be small and harder to detect")
    print("- Other unmeasured factors may be more important")
else:
    for var, row in significant_vars.iterrows():
        odds_ratio = row['Odds Ratio']
        p_val = row['P-value']
        
        print(f"\n{var}:")
        print(f"  Odds Ratio: {odds_ratio:.3f} (p = {p_val:.4f})")
        
        if odds_ratio > 1:
            pct_change = (odds_ratio - 1) * 100
            print(f"POSITIVE effect: {pct_change:.1f}% increase in odds of winning")
            
            if 'genre_' in var:
                genre_name = var.replace('genre_', '')
                print(f"  Films in {genre_name} genre have {pct_change:.1f}% higher odds")
                print(f"  of winning compared to {reference_genre} (reference category)")
                print(f"  Interpretation: Being in the {genre_name} genre significantly")
                print(f"  increases Oscar win probability, even after controlling for")
                print(f"  film quality, popularity, and runtime.")
            elif var == 'averageRating':
                print(f"  Each 1-point increase in IMDb rating increases winning odds by {pct_change:.1f}%")
                print(f"  Interpretation: Higher-rated films are significantly more likely to win,")
                print(f"  suggesting quality metrics predict Academy preferences.")
            elif var == 'log_votes':
                print(f"  Higher vote counts (popularity) increase winning odds")
                print(f"  Interpretation: More popular films (measured by voting volume)")
                print(f"  have higher win probability, suggesting visibility matters.")
            elif var == 'runtimeMinutes':
                print(f"  Each additional minute increases winning odds by {pct_change:.1f}%")
                print(f"  Interpretation: Longer films have an advantage, possibly reflecting")
                print(f"  perceived epic scope or artistic ambition.")
            elif var == 'era':
                print(f"  Post-reform era (2015-2024) has {pct_change:.1f}% higher winning odds")
                print(f"  compared to pre-reform era (2004-2014)")
                print(f"  Interpretation: Academy reforms changed winning patterns,")
                print(f"  possibly due to diversified membership or shifting preferences.")
        else:
            pct_change = (1 - odds_ratio) * 100
            print(f"NEGATIVE effect: {pct_change:.1f}% decrease in odds of winning")
            
            if 'genre_' in var:
                genre_name = var.replace('genre_', '')
                print(f"  Films in {genre_name} genre have {pct_change:.1f}% lower odds")
                print(f"  of winning compared to {reference_genre} (reference category)")
                print(f"  Interpretation: Being in the {genre_name} genre significantly")
                print(f"  decreases Oscar win probability, suggesting genre bias against")
                print(f"  this category even after controlling for quality.")
            elif var == 'era':
                print(f"  Post-reform era (2015-2024) has {pct_change:.1f}% lower winning odds")
                print(f"  Interpretation: Despite reforms, later era shows lower win")
                print(f"  probability, possibly due to increased competition or changing standards.")

print()

# Model fit statistics
print("\nModel Fit Statistics:")
print("-"*80)
print(f"Log-Likelihood: {result.llf:.4f}")
print(f"AIC: {result.aic:.4f}")
print(f"BIC: {result.bic:.4f}")
print(f"Pseudo R-squared (McFadden): {result.prsquared:.4f}")
print()

# Interpret model fit
if result.prsquared < 0.2:
    print("Pseudo R^2 interpretation: Low explanatory power")
    print("  The model explains some variation but many factors remain unexplained.")
elif result.prsquared < 0.4:
    print("Pseudo R^2 interpretation: Moderate explanatory power")
    print("  The model captures meaningful patterns in Oscar winning.")
else:
    print("Pseudo R^2 interpretation: High explanatory power")
    print("  The model explains substantial variation in winning probability.")
print()

# Calculate accuracy on training data
predicted_probs = result.predict(X_with_const)
predicted_class = (predicted_probs > 0.5).astype(int)
accuracy = (predicted_class == y).mean()
print(f"Classification Accuracy: {accuracy*100:.2f}%")

# Baseline accuracy (always predict majority class)
baseline_accuracy = max((y == 0).mean(), (y == 1).mean())
print(f"Baseline Accuracy (majority class): {baseline_accuracy*100:.2f}%")

if accuracy > baseline_accuracy:
    improvement = (accuracy - baseline_accuracy) * 100
    print(f"Improvement over baseline: +{improvement:.2f} percentage points")
else:
    print("Model does not improve over baseline prediction.")
print()

In [ ]:
# Plot odds ratios with confidence intervals
fig, ax = plt.subplots(figsize=(10, 8))

# Prepare data (exclude intercept)
plot_data = coef_df.drop('const', errors='ignore').sort_values('Odds Ratio')

y_pos = np.arange(len(plot_data))

# Plot odds ratios as points with error bars
ax.errorbar(plot_data['Odds Ratio'], y_pos, 
            xerr=[plot_data['Odds Ratio'] - plot_data['95% CI Lower'],
                  plot_data['95% CI Upper'] - plot_data['Odds Ratio']],
            fmt='o', markersize=8, capsize=5, capthick=2,
            ecolor='gray', elinewidth=2)

# Color points by direction of effect
for i, (idx, row) in enumerate(plot_data.iterrows()):
    color = 'green' if row['Odds Ratio'] > 1 else 'red'
    ax.plot(row['Odds Ratio'], i, 'o', markersize=10, color=color)

    # Add asterisk for significant predictors
    if row['P-value'] < 0.05:
        ax.text(row['Odds Ratio'], i + 0.3, '*', 
                fontsize=20, ha='center', fontweight='bold')

# Reference line at 1 (no effect)
ax.axvline(x=1, color='black', linestyle='--', linewidth=2, label='No Effect (OR=1)')

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_data.index)
ax.set_xlabel('Odds Ratio (with 95% CI)', fontsize=12)
ax.set_title('Logistic Regression: Odds Ratios for Winning an Oscar', 
             fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='x', alpha=0.3)

# Add text annotation
ax.text(0.02, 0.98, '* = p < 0.05 (statistically significant)',
        transform=ax.transAxes, fontsize=10,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.show()

print()
